# Peer-pricing pipeline: a local, inspectable walkthrough

This demo follows source CSVs -> macro lookup -> harmonized corpus -> train-only fitted metadata/text -> 53D coordinates -> nearest peers -> corridor and fee -> exported artifacts -> local API. It reuses the project modules rather than replacing their business logic.

**Not production or accuracy evidence:** marketplace rows below are invented; macro inputs are small public repository test fixtures, not current economic data. Proxy budgets/asking rates are not verified mentor payments. Synthetic timestamps are not genuine source observations. We do not run or claim to pass the deferred empirical OOT gate (#81/#91). Firebase, Firestore, credentials and network latency measurement are deliberately excluded.

Read [the overview](../README.md), [glossary](../docs/README.md), [historical blueprint](../docs/Blueprint.md) and [provenance audit](../docs/Label_Provenance_Audit_91.md) alongside this notebook.

## 1. Setup and safety

Use Python 3.11+ and a kernel from the environment containing `requirements.txt`. From the repository root in PowerShell:

```powershell
python -m pip install -r requirements.txt
python -m pip install jupyterlab ipykernel
python -m nltk.downloader stopwords wordnet
python -m jupyterlab notebooks\pipeline_walkthrough.ipynb
```

Select that Python kernel and Run All, or step through each cell. Run from the repository root or `notebooks` directory. The setup fails with an actionable message for missing required packages/NLTK resources rather than attempting a network download or silently substituting preprocessing. Optional Firebase services are never initialized.

All generated CSVs, parquet and fitted artifacts go into one OS temporary directory, **not `data/`, `Data/` or the repository artifact directory**. The last cell removes it; if you interrupt, run that cell or restart the kernel. Re-running setup cleans its previous temporary workspace. This small 128-row demo keeps training bounded; no full local datasets or long-running research reports are loaded.

In [ ]:
import importlib.util
import json
from pathlib import Path
import sys
import tempfile

candidates = (Path.cwd(), Path.cwd().parent)
ROOT = next((p for p in candidates if p.joinpath('src', 'train_pipeline.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Start the notebook from the repository root or notebooks directory.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
required = ('pandas', 'numpy', 'pyarrow', 'pycountry', 'nltk', 'sklearn', 'joblib', 'fastapi', 'httpx', 'pydantic')
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(f'Missing {missing}; install requirements.txt into this notebook kernel.')

import nltk
from nltk.corpus import stopwords, wordnet
try:
    stopwords.words('english')
    wordnet.ensure_loaded()
except LookupError as exc:
    raise RuntimeError('Run python -m nltk.downloader stopwords wordnet in the kernel environment, then rerun.') from exc

import numpy as np
import pandas as pd
from IPython.display import display
from fastapi.testclient import TestClient
from src.ingest_multisource import (
    build_macro_lookup, load_macro_lookup_table, build_harmonized_marketplace_records,
    export_harmonized_records_parquet, parse_upwork_jobs_dataset, marketplace_exclusion_reason,
)
from src.train_pipeline import load_harmonized_parquet, build_stratified_splits, evaluate_and_serialize_training
from src.macro_arbitrage import ContinuousMetadataNormalizer, DEFAULT_FEATURE_NAMES, fuse_coordinate_batches, fuse_coordinates
from src.nlp_pipeline import TextFeatureReducer, sanitize_many, ensure_text_dimensions
from src.spatial_engine import DomainPartitionedKDTreeIndexer
from src.interval_synthesizer import synthesize_corridor
from src.tariff_evaluator import MpesaTariffEvaluator, DEFAULT_MPESA_TARIFF_CSV
from src.artifact_contract import file_sha256, verify_manifest
from src.api_contracts import CanonicalPricingQueryDTO
from src.repository import InMemoryRepository
from src.serve import InferenceRuntime, create_app

if 'workspace' in globals():
    workspace.cleanup()
workspace = tempfile.TemporaryDirectory(prefix='price-notebook-')
WORK = Path(workspace.name)
macro_path = WORK.joinpath('macro_lookup_table.json')
parquet_path = WORK.joinpath('demo_corpus.parquet')
artifact_dir = WORK.joinpath('export')
print({'repository': str(ROOT), 'temporary_workspace': str(WORK), 'Firebase': 'not used'})

## 2. Generate and reload the macro lookup

`build_macro_lookup` reads WDI PPP, cost-index and historical tariff-summary CSV fixtures. PPP is local currency per international dollar; cost-of-living is an index, not an hourly rate. The saved lookup retains reference years, source filenames and fallback flags. Its historical tariff summary is **not** the current quote tariff: the quote stage uses the versioned 15-band P2P schedule from `config`.

In [ ]:
fixture_root = ROOT.joinpath('tests', 'fixtures', 'raw')
macro_payload = build_macro_lookup(str(fixture_root), str(macro_path))
macro_records = load_macro_lookup_table(str(macro_path))
display(pd.DataFrame.from_dict(macro_records, orient='index')[
    ['country_iso2', 'ppp_lcu_per_intl_dollar', 'ppp_reference_year', 'cost_of_living_index', 'cost_index_fallback_used']
].head(8))
print({'lookup_file': macro_path.name, 'countries': len(macro_records), 'inputs': 'read-only test fixtures'})

## 3. Invent source rows, then call the real ingestion path

We create source-shaped Upwork job/profile CSVs **in temporary storage**. The alphabetic tags provide enough distinct vocabulary for the fitted 50-component demonstration. Rates are invented USD/hour; timestamps on jobs are explicitly artificial UTC publication dates. Profile rows have no observation date and retain null dates.

The parser ignores non-hourly jobs. The shared gate rejects descriptions shorter than 40 characters or pre-arbitrage rates outside 500-35,000 KES/hour (and invalid rates/industry). Harmonization converts USD/hour using the module's fixed anchor, normalizes countries, applies the bilateral factor and derives corpus-frequency metadata. It does not fetch or verify mentor transactions. One deliberate short-description row below makes an exclusion visible.

In [ ]:
raw_dir = WORK.joinpath('raw')
jobs_dir = raw_dir.joinpath('upwork-jobs.csv')
profiles_dir = raw_dir.joinpath('Data_Scientist_Upwork')
jobs_dir.mkdir(parents=True)
profiles_dir.mkdir(parents=True)
jobs = []
for i in range(120):
    title = 'Python data scientist analytics mentor' if i < 72 else 'Backend web developer javascript mentor'
    tag = 'topic' + chr(97 + i // 26) + chr(97 + i % 26)
    jobs.append({
        'title': title,
        'description': f'{title} offers practical mentoring for learners and teams through clear project reviews, testing guidance, skill development and detailed technical explanations. {tag} {tag}workshop',
        'country': 'Kenya' if i % 2 == 0 else 'United States',
        'is_hourly': True, 'hourly_low': 12 + i % 12, 'hourly_high': 16 + i % 12,
        'published_date': f'2025-01-{1 + i % 28:02d}T12:00:00Z',
    })
jobs.append({**jobs[0], 'description': 'too short'})
jobs_path = jobs_dir.joinpath('upwork-jobs.csv')
pd.DataFrame(jobs).to_csv(jobs_path, index=False)
profiles = [{
    'title': 'Data scientist mentor', 'skills': 'python analytics', 'country': 'Kenya', 'hourlyRate': 25 + i,
    'description': f'Python analytics mentoring with careful code review, clear explanations and practical project exercises for learners and technical teams. profiletopic{chr(97 + i)}',
} for i in range(8)]
pd.DataFrame(profiles).to_csv(profiles_dir.joinpath('upwork_data_scientists.csv'), index=False)
staged = parse_upwork_jobs_dataset(str(jobs_path))
display(pd.DataFrame([{'description': row['raw_description'][:65], 'exclusion': marketplace_exclusion_reason(row)}
                      for row in (staged[0], staged[-1])]))
records = build_harmonized_marketplace_records(str(raw_dir), str(macro_path))
export_harmonized_records_parquet(records, str(parquet_path))
frame = load_harmonized_parquet(str(parquet_path))
assert len(frame) == 128
display(frame[['source_dataset', 'industry_partition', 'client_country_iso2', 'hourly_rate_usd',
               'hourly_rate', 'bilateral_arbitrage_factor', 'harmonized_hourly_rate', 'observation_timestamp_utc']].head(6))
print({'accepted_rows': len(frame), 'missing_source_dates': int(frame.observation_timestamp_utc.isna().sum()),
       'target_unit': 'KES/hour', 'label_provenance': 'synthetic proxy, not mentor payments'})

## 4. Split before fitting; normalize the three metadata columns

This is seeded **stratified random exploratory** evaluation, not chronological OOT. Only training rows fit the MinMax scaler, vocabulary, SVD and peer index. The legacy stratified path uses corpus-derived frequency inputs; the separate OOT path replaces these with train-only counts. No target-rate column becomes a feature.

Current metadata order: bilateral country factor, market saturation, industry-relative density. An optional request cost-of-living value modifies the country factor, not a fourth metadata coordinate. `metadata_enrichment.py` supports optional offline ablation groups and is not part of this production 53D schema.

In [ ]:
split = build_stratified_splits(frame, random_state=42)
train_records = split.train.to_dict(orient='records')
normalizer = ContinuousMetadataNormalizer(str(macro_path))
train_meta = normalizer.fit_transform(train_records)
validation_meta = normalizer.transform(split.validation.to_dict(orient='records'))
test_meta = normalizer.transform(split.test.to_dict(orient='records'))
display(pd.DataFrame({'train_min': normalizer.scaler.data_min_, 'train_max': normalizer.scaler.data_max_},
                     index=DEFAULT_FEATURE_NAMES))
display(pd.DataFrame(train_meta[:5], columns=DEFAULT_FEATURE_NAMES))
print({'split_rows': {name: len(getattr(split, name)) for name in ('train', 'validation', 'test')},
       'metadata_shapes': [train_meta.shape, validation_meta.shape, test_meta.shape]})

## 5. Inspect sanitization, TF-IDF and text reduction

The reducer sanitizes text, fits weighted word/word-pair features, then fits SVD. Validation and test only transform through those frozen components. `ensure_text_dimensions` enforces the 50-column text contract; small corpora can yield fewer latent axes and are padded. These coordinates are not named skills or an accuracy score.

In [ ]:
train_texts = split.train.raw_description.tolist()
reducer = TextFeatureReducer(n_components=50, random_state=42)
train_text = ensure_text_dimensions(reducer.fit_transform(train_texts))
validation_text = ensure_text_dimensions(reducer.transform(split.validation.raw_description.tolist()))
test_text = ensure_text_dimensions(reducer.transform(split.test.raw_description.tolist()))
cleaned = sanitize_many(train_texts[:2])
display(pd.DataFrame({'raw': train_texts[:2], 'sanitized': cleaned}))
tfidf_preview = reducer.vectorizer.transform(cleaned)
print({'TFIDF_preview_shape': tfidf_preview.shape, 'nonzero_entries': tfidf_preview.nnz,
       'SVD_fitted_components': reducer.reducer.n_components, 'text_shapes': [train_text.shape, validation_text.shape, test_text.shape],
       'explained_variance_sum_not_accuracy': reducer.explained_variance_sum()})
display(pd.DataFrame(train_text[:3, :8], columns=[f'text_{i}' for i in range(8)]))

## 6. Fuse coordinates and retrieve peer rates

Fusion is `[50 text | bilateral factor, saturation, density]`: exactly 53 columns. The index stores training row IDs; its internal partition positions are mapped back to those IDs. Closer vectors have larger normalized quadratic inverse-distance weights. The API name `verified_rate` is historical: these rates remain invented proxies. No listing IDs are invented, so canonical comparables will be empty.

In [ ]:
x_train = fuse_coordinate_batches(train_text, train_meta)
x_validation = fuse_coordinate_batches(validation_text, validation_meta)
x_test = fuse_coordinate_batches(test_text, test_meta)
assert x_train.shape[1] == 53
indexer = DomainPartitionedKDTreeIndexer(minimum_partition_size=5)
indexer.fit(x_train, split.train.industry_partition.tolist(),
            record_indices=split.train.record_id.tolist(), verified_rates=split.train.target_rate.tolist())
sample = split.validation.iloc[0]
peer_prediction = indexer.predict_base_rate(x_validation[0], sample.industry_partition, k=5)
display(pd.DataFrame(peer_prediction['nearest_neighbors']))
print({'hybrid_shapes': [x_train.shape, x_validation.shape, x_test.shape],
       'partition_counts': indexer.partition_counts, 'requested': sample.industry_partition,
       'routed': peer_prediction['routed_partition'], 'base_KES_per_hour': peer_prediction['base_predicted_rate'],
       'peer_stddev_KES_per_hour': peer_prediction['peer_stddev']})

## 7. Calculate the fee-free corridor, then add the approved-rail fee

The floor affects the lower negotiation bound, not the peer base. The configured 15-band transfer-to-M-PESA-user schedule adds one Kenyan fee to the point quote; other rails are reference-only. The final quote can exceed the corridor maximum. Unsupported Kenyan amounts fail rather than being clamped/extrapolated. No money is transferred. This local demonstration reads the checked-in tariff; deployment requires a separately approved hash and source review.

In [ ]:
tariff_path = ROOT.joinpath(DEFAULT_MPESA_TARIFF_CSV)
tariff = MpesaTariffEvaluator.from_csv(str(tariff_path))
corridor = synthesize_corridor(peer_prediction['base_predicted_rate'], peer_prediction['peer_stddev'],
                               float(sample.bilateral_arbitrage_factor))
quote = tariff.evaluate_quote(peer_prediction['base_predicted_rate'], 'KE')
assert len(tariff.quote_bands) == 15
assert quote['final_quoted_rate'] == round(quote['base_predicted_rate'] + quote['mpesa_tariff_surcharge'], 2)
display(pd.DataFrame([{**corridor, **quote}]).drop(columns=['applied_tariff_band']))
print({'applied_band': quote['applied_tariff_band'], 'fee_free_corridor': True})

## 8. Run the training entrypoint and inspect its export

`evaluate_and_serialize_training` repeats the same seeded training using the production source path and writes a complete manifest-compatible bundle. It reports local random-split metrics and a category-mean baseline. We explicitly disable historical stratified promotion checks **only for this temporary exploratory export**. A numerical score here says nothing about real mentor-price accuracy and does not pass the deferred OOT gate.

Hashing is allowed here solely because we just built this synthetic bundle in our own process. **Never auto-pin an arbitrary downloaded joblib bundle**: hashes are integrity checks, not authenticity. Deployment obtains its manifest and tariff pins from independently trusted release records.

In [ ]:
summary = evaluate_and_serialize_training(
    str(parquet_path), str(macro_path), artifact_dir=str(artifact_dir), random_state=42,
    minimum_partition_size=5, enforce_quality_gate=False, dataset_version='synthetic-notebook-v1',
)
manifest_pin = file_sha256(str(artifact_dir.joinpath('artifact_manifest.json')))
manifest = verify_manifest(str(artifact_dir), manifest_pin)
display(pd.DataFrame(summary['model_metrics']).T)
display(pd.DataFrame(summary['baseline_metrics']).T)
print({'protocol': summary['evaluation_protocol'], 'status': summary['validation_status'],
       'quality_gate': summary['quality_gate'], 'provenance': manifest['provenance']})
display(pd.DataFrame([{'file': p.name, 'bytes': p.stat().st_size} for p in sorted(artifact_dir.iterdir())]))

## 9. Reload the trusted local export and compare inference stages

A fresh runtime checks the manifest before deserializing fitted artifacts. Its load failure is surfaced explicitly, not hidden by a fake prediction. Request metadata uses the frozen export density mapping and scaler; request text never refits the vocabulary. The optional cost-of-living override is omitted here. We compare the source-stage base estimate with the reloaded runtime to expose train/serve wiring.

In [ ]:
runtime = InferenceRuntime(str(artifact_dir), str(macro_path), str(tariff_path),
                           trusted_manifest_sha256=manifest_pin)
runtime.load()
if not runtime.models_loaded:
    raise RuntimeError(f'Local export failed readiness: {runtime.load_error}')
request = CanonicalPricingQueryDTO.model_validate({
    'mentorId': 'demo-mentor', 'rawText': sample.raw_description,
    'industry': sample.industry_partition, 'mentorCountry': 'KE',
    'clientCountry': sample.client_country_iso2, 'competitivenessScore': float(sample.market_saturation_score),
})
query = request.to_inference_query()
live_meta = normalizer.transform_live_metadata('KE', query.client_country, query.market_saturation_score,
                                              runtime.inference_config['partition_density'][query.selected_industry])
live_vector = fuse_coordinates(ensure_text_dimensions(reducer.transform([query.raw_description])), live_meta)
stage_prediction = indexer.predict_base_rate(live_vector, query.selected_industry, k=5)
runtime_quote = runtime.predict(query)
assert runtime_quote.base_predicted_rate == stage_prediction['base_predicted_rate']
display(pd.DataFrame([runtime_quote.model_dump(exclude={'nearest_neighbors'})]))
print({'models_loaded': runtime.models_loaded, 'stage_reload_base_parity': True,
       'validation_status': runtime.manifest['provenance']['validation_status']})

## 10. Exercise local readiness, canonical API and background audit

`TestClient` runs the real app in-process with actual artifact loading and an `InMemoryRepository`; there is no listening network server. `/ready` is expected to return 200, the canonical quote must preserve the reloaded result, and one background audit is visible after the successful request. Proxy rows produce no canonical listing comparables and confidence remains 0.0 (uncalibrated).

This app deliberately has **no token verifier** and must not be deployed. `src.deployment:app` instead requires Firebase verification, Firestore mentor `auth_uid` ownership/country checks, backend readiness, externally pinned artifacts and a reviewed tariff pin. The notebook does not initialize Firebase or emulate successful protected authorization. See [deployment notes](../docs/M13.2_Container_and_CI.md), [audit limitations](../docs/M11.2_Pricing_Audit.md) and [in-process benchmark caveats](../docs/M10.3_Protected_Pricing_Latency.md). This is neither real mobile RTT (#86) nor an SLA.

In [ ]:
repository = InMemoryRepository()
app = create_app(artifact_dir=str(artifact_dir), macro_lookup_path=str(macro_path),
                 tariff_csv_path=str(tariff_path), repository=repository, trusted_manifest_sha256=manifest_pin)
with TestClient(app) as client:
    health = client.get('/health')
    ready = client.get('/ready')
    assert health.status_code == ready.status_code == 200, ready.text
    response = client.post('/api/v1/optimize-price', json=request.model_dump(by_alias=True, exclude_none=True))
    assert response.status_code == 200, response.text
    result = response.json()
    assert result['basePredictedRate'] == runtime_quote.base_predicted_rate
    assert result['confidenceScore'] == 0.0 and result['comparables'] == []
    assert len(repository.list_transactions()) == 1
    print(json.dumps({'health': health.json(), 'quote': result}, indent=2))
display(pd.DataFrame(repository.list_transactions()))
print('Local quote and in-memory audit only: no Firebase, Firestore, external credentials or network RTT.')

## 11. Cleanup and production boundaries

The production source entrypoints remain `python -m src.ingest_multisource`, `python -m src.train_pipeline` and the local/protected ASGI apps. This notebook is an educational runner, not an alternative production pipeline or a release record. It leaves no datasets, binaries or reports in Git; notebook outputs are intentionally not saved in the committed file. After cleanup, restart from setup to run again.

Real release work needs reviewed source provenance and consent, immutable artifact/tariff pins, configured authorization/storage and deployment integration. Genuine timestamp-complete observations and a predeclared chronological split are necessary for optional OOT research; our invented dates and local metrics cannot establish those facts.

In [ ]:
workspace.cleanup()
assert not WORK.exists()
print('Temporary inputs and exports removed. No repository data directories were written.')